In [1]:
from zipfile import ZipFile
import requests
import os 
import io
import datetime
import dateutil
import time
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/"

package_show_url = base_url + "package_show?id=fire-incidents"
package_response = requests.get(package_show_url).json()

resources = package_response["result"]["resources"]
resource_id = [r["id"] for r in resources if r["format"] == "CSV"][0]

datastore_url = f"https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/{resource_id}"
df_fire = pd.read_csv(datastore_url)
df_fire

C:\Users\Elowe\AppData\Local\Temp\ipykernel_29628\2591964713.py:10: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fire = pd.read_csv(datastore_url)


,_id,Incident_Number,Initial_CAD_Event_Type,Final_Incident_Type,Exposures,Incident_Station_Area,Incident_Ward,Intersection,Latitude,Longitude,...,Possible_Cause,Smoke_Alarm_at_Fire_Origin,Smoke_Alarm_at_Fire_Origin_Alarm_Failure,Smoke_Alarm_at_Fire_Origin_Alarm_Type,Smoke_Alarm_Impact_on_Persons_Evacuating_Impact_on_Evacuation,Fire_Alarm_System_Presence,Fire_Alarm_System_Operation,Fire_Alarm_System_Impact_on_Evacuation,Sprinkler_System_Presence,Sprinkler_System_Operation
0,1,F18020956,Vehicle Fire,01 - Fire,NaN,441,1.0,Dixon Rd / 427 N Dixon Ramp,43.686558,-79.599419,...,99 - Undetermined,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,F18020969,Fire - Grass/Rubbish,01 - Fire,NaN,116,18.0,Sheppard Ave E / Clairtrell Rd,43.766135,-79.390039,...,03 - Suspected Vandalism,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,F18021182,Fire - Highrise Residential,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...",NaN,221,21.0,Danforth Rd / Savarin St,43.743230,-79.245061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,F18021192,Fire - Commercial/Industrial,01 - Fire,NaN,133,5.0,Keele St / Lawrence Ave W,43.708659,-79.478062,...,99 - Undetermined,9 - Floor/suite of fire origin: Smoke alarm pr...,98 - Not applicable: Alarm operated OR presenc...,9 - Type undetermined,"8 - Not applicable: No alarm, no persons present",9 - Undetermined,8 - Not applicable (no system),9 - Undetermined,9 - Undetermined,8 - Not applicable - no sprinkler system present
4,5,F18021271,Fire - Residential,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...",NaN,132,8.0,Replin Rd / Tapestry Lane,43.718118,-79.443184,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36559,36560,F24060535,FIG,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...",0.0,334,10.0,Spadina Ave / Lake Shore Blvd W / Lower Spadin...,43.638454,-79.392226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36560,36561,F24126112,FIG2,01 - Fire,0.0,334,10.0,Spadina Ave / Lake Shore Blvd W / Lower Spadin...,43.638454,-79.392226,...,Other Unintended Cause,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36561,36562,F24024036,FIG,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...",0.0,334,10.0,Spadina Ave / Lake Shore Blvd W / Lower Spadin...,43.638454,-79.392226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36562,36563,F24044914,FIG,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...",0.0,332,10.0,Stephanie St / John St,43.651197,-79.391623,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_fire['Alarm_Time'] = pd.to_datetime(df_fire['TFS_Alarm_Time'])
df_fire['Arrival_Time'] = pd.to_datetime(df_fire['TFS_Arrival_Time'])
df_fire['Duration'] = (df_fire['Arrival_Time'] - df_fire['Alarm_Time']).dt.total_seconds()
df_fire['Hourly_Timestamp'] = df_fire['Alarm_Time'].dt.floor('h')
print(df_fire[['Incident_Station_Area', 'Duration']])

                Incident_Station_Area  Duration
0                                 441     342.0
1                                 116     288.0
2                                 221     410.0
3                                 133     268.0
4                                 132     336.0
...                               ...       ...
36559  334                                489.0
36560  334                                 94.0
36561  334                                263.0
36562  332                                327.0
36563  313                                210.0

[36564 rows x 2 columns]


This dataset contains fire incidents in Toronto from 2011-01-01 to 2024-12-31. Additionally, I merge it with the weather data since the weather my impact the response of the fire incidence as well.

In [4]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 43.65,
    "longitude": -79.38,
    "start_date": "2011-01-01",
    "end_date": "2025-01-01",
    "hourly": "temperature_2m,snowfall,precipitation",
    "timezone": "America/New_York"
}

weather_res = requests.get(weather_url, params=params).json()
df_weather = pd.DataFrame(weather_res['hourly'])
df_weather['time'] = pd.to_datetime(df_weather['time'])


df_merged = pd.merge(
    df_fire, 
    df_weather, 
    left_on='Hourly_Timestamp', 
    right_on='time', 
    how='left'
)

In [5]:
time_cols = ['TFS_Alarm_Time', 'TFS_Arrival_Time', 'Fire_Under_Control_Time', 'Hourly_Timestamp']
for col in time_cols:
    df_fire[col] = pd.to_datetime(df_fire[col])

In [6]:
df_merged.isnull().sum()

_id                                                                  0
Incident_Number                                                      0
Initial_CAD_Event_Type                                               1
Final_Incident_Type                                                  0
Exposures                                                        25418
Incident_Station_Area                                                1
Incident_Ward                                                      158
Intersection                                                         2
Latitude                                                             2
Longitude                                                            2
Property_Use                                                       224
Building_Status                                                  18483
TFS_Alarm_Time                                                       0
TFS_Arrival_Time                                                     1
Ext_ag

In [7]:
df_merged.to_csv("fire.csv", index=False)